In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

In [2]:
#Abrindo os arquivos Excel

vendas_df = pd.read_excel(r"D:\Programação\dashboard-vendas-multi-loja\planilhas_vendas\Vendas.xlsx",sheet_name='Plan1')

vendas_dz_df = pd.read_excel(r"D:\Programação\dashboard-vendas-multi-loja\planilhas_vendas\Vendas - Dez.xlsx", sheet_name='Plan1')

gerentes_df = pd.read_excel(r"D:\Programação\dashboard-vendas-multi-loja\planilhas_vendas\Gerentes.xlsx", sheet_name='Planilha1')

In [3]:
#juntar as tabelas de vendas
vendas_completo = pd.concat([vendas_df, vendas_dz_df], ignore_index=True)

#juntar a tabela de vendas com a tabela de gerentes
vendas_completo = vendas_completo.merge(gerentes_df, on="ID Loja", how="left")

#Tratar tabela de vendas
vendas_completo = vendas_completo.drop_duplicates()

vendas_completo = vendas_completo.dropna(subset=["Valor Final"])

vendas_completo["Data"] = pd.to_datetime(vendas_completo["Data"])

In [4]:
# Criar colunas de Ano, Mês e Semana
vendas_completo["Ano"] = vendas_completo["Data"].dt.year
vendas_completo["Mes"] = vendas_completo["Data"].dt.month

vendas_completo["Ano_Mes"] = vendas_completo["Data"].dt.to_period("M")

# Mês em português
mapa_meses = {
    1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
    5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
    9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
}

vendas_completo["Mes_Nome"] = vendas_completo["Mes"].map(mapa_meses)

# Dia da semana em português
mapa_dias = {
    "Monday": "Segunda-feira",
    "Tuesday": "Terça-feira",
    "Wednesday": "Quarta-feira",
    "Thursday": "Quinta-feira",
    "Friday": "Sexta-feira",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

vendas_completo["Dia_Semana"] = vendas_completo["Data"].dt.day_name().map(mapa_dias)

# Ordenar colunas
colunas_data = ["Data", "Ano", "Mes", "Mes_Nome", "Ano_Mes", "Dia_Semana"]

outras_colunas = [col for col in vendas_completo.columns if col not in colunas_data]

vendas_completo = vendas_completo[colunas_data + outras_colunas]

display(vendas_completo)

,Data,Ano,Mes,Mes_Nome,Ano_Mes,Dia_Semana,Código Venda,ID Loja,Produto,Quantidade,Valor Unitário,Valor Final,Gerente
0,2019-01-01,2019,1,Janeiro,2019-01,Terça-feira,1,Iguatemi Esplanada,Sapato Estampa,1,358,358,Salvador
1,2019-01-01,2019,1,Janeiro,2019-01,Terça-feira,1,Iguatemi Esplanada,Camiseta,2,180,360,Salvador
2,2019-01-01,2019,1,Janeiro,2019-01,Terça-feira,1,Iguatemi Esplanada,Sapato Xadrez,1,368,368,Salvador
3,2019-01-02,2019,1,Janeiro,2019-01,Quarta-feira,2,Norte Shopping,Relógio,3,200,600,Joana
4,2019-01-02,2019,1,Janeiro,2019-01,Quarta-feira,2,Norte Shopping,Chinelo Liso,1,71,71,Joana
...,...,...,...,...,...,...,...,...,...,...,...,...,...
100994,2019-12-26,2019,12,Dezembro,2019-12,Quinta-feira,69996,Center Shopping Uberlândia,Short Listrado,2,102,204,Andressa
100995,2019-12-26,2019,12,Dezembro,2019-12,Quinta-feira,69996,Center Shopping Uberlândia,Mochila,4,270,1080,Andressa
100996,2019-12-26,2019,12,Dezembro,2019-12,Quinta-feira,69996,Center Shopping Uberlândia,Pulseira Estampa,1,87,87,Andressa
100997,2019-12-26,2019,12,Dezembro,2019-12,Quinta-feira,69997,Ribeirão Shopping,Camisa Listrado,1,108,108,Fábio


In [5]:
faturamento_total = vendas_completo["Valor Final"].sum()

faturamento_loja = vendas_completo.groupby("ID Loja")["Valor Final"].sum().sort_values(ascending=False)

faturamento_gerente = vendas_completo.groupby("Gerente")["Valor Final"].sum().sort_values(ascending=False)

produtos_mais_vendidos = vendas_completo.groupby("Produto")["Quantidade"].sum().sort_values(ascending=False)

ticket_medio_loja = vendas_completo.groupby("ID Loja")["Valor Final"].mean().sort_values(ascending=False)

# Análise por data
analise_dia = vendas_completo.groupby("Data")["Valor Final"].sum().sort_index()

ordem = [
    "Segunda-feira", "Terça-feira", "Quarta-feira",
    "Quinta-feira", "Sexta-feira", "Sábado", "Domingo"
]

analise_dia_semana = (
    vendas_completo.groupby("Dia_Semana")["Valor Final"]
    .sum()
    .reindex(ordem)
)

# CORREÇÃO AQUI: Remover o reset_index() para manter como Series ou preparar para o gráfico
analise_mes = (
    vendas_completo
    .groupby(["Ano", "Mes", "Mes_Nome"])["Valor Final"]
    .sum()
    .reset_index()  # Mantém como DataFrame
)

analise_ano = vendas_completo.groupby("Ano")["Valor Final"].sum().sort_index()

# Exibir as análises
display(analise_dia)
display(analise_dia_semana)
display(analise_mes)
display(analise_ano)

Data
2019-01-01      1086
2019-01-02    110995
2019-01-03    122168
2019-01-04    129934
2019-01-05    124824
               ...  
2019-12-22    110931
2019-12-23    128342
2019-12-24    110618
2019-12-25    113907
2019-12-26     21591
Name: Valor Final, Length: 360, dtype: int64

Dia_Semana
Segunda-feira    6010393
Terça-feira      5945169
Quarta-feira     6071583
Quinta-feira     5984632
Sexta-feira      5974931
Sábado           5869236
Domingo          5970094
Name: Valor Final, dtype: int64

,Ano,Mes,Mes_Nome,Valor Final
0,2019,1,Janeiro,3462957
1,2019,2,Fevereiro,3274977
2,2019,3,Março,3642537
3,2019,4,Abril,3547336
4,2019,5,Maio,3600535
5,2019,6,Junho,3540377
6,2019,7,Julho,3621529
7,2019,8,Agosto,3543707
8,2019,9,Setembro,3491005
9,2019,10,Outubro,3649728


Ano
2019    41826038
Name: Valor Final, dtype: int64

In [6]:
valores_formatados_top10 = [
    f"{v:,.0f}".replace(",", ".")
    for v in faturamento_loja.head(10).values
]
valores_formatados_piores = [
    f"{v:,.0f}".replace(",", ".")
    for v in faturamento_loja.tail(5).values
]

# GRÁFICO PRINCIPAL: Top 10 melhores
fig1 = px.bar(
    x=faturamento_loja.head(10).index,
    y=faturamento_loja.head(10).values,
    text=valores_formatados_top10, 
    title="TOP 10 LOJAS COM MAIOR FATURAMENTO",
    labels={'x': 'Loja', 'y': 'Faturamento (R$)'},
    color=faturamento_loja.head(10).values,
    color_continuous_scale='Greens',
    height=600  
)

fig1.update_traces(
    textposition='outside',
    textfont=dict(size=11, color='white'),
    marker=dict(line=dict(color='white', width=2)),
    customdata=valores_formatados_top10,
    hovertemplate="Loja: %{x}<br>Faturamento: R$ %{customdata}<extra></extra>"
)

fig1.update_yaxes(
    range=[0, faturamento_loja.head(10).max() * 1.15],
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    showgrid=True,
    gridcolor='gray',
    griddash='dot',
    ticklabelposition="outside",
    ticklabelstandoff=12,
    ticks="outside",
    title_standoff=25,
    title_font=dict(size=14, color='white')
)

fig1.update_xaxes(
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    tickangle=35,  
    tickfont=dict(size=9, color='white'),  
    title_font=dict(size=14, color='white'),
    ticklabelposition="outside"
)

fig1.update_layout(
    margin=dict(t=80, l=120, r=50, b=140),  
    height=600,
    bargap=0.35,
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white', size=12),
    title_font=dict(size=22, color='white'),
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    ),
    coloraxis_showscale=False
)

fig1.show()

# GRÁFICO COMPLEMENTAR: Top 5 piores
fig_piores = px.bar(
    x=faturamento_loja.tail(5).index,
    y=faturamento_loja.tail(5).values,
    text=valores_formatados_piores,
    title="TOP 5 LOJAS COM MENOR FATURAMENTO",
    labels={'x': 'Loja', 'y': 'Faturamento (R$)'},
    color=faturamento_loja.tail(5).values,
    color_continuous_scale='Reds',
    height=550  
)

fig_piores.update_traces(
    textposition='outside',
    textfont=dict(size=12, color='white'),  
    customdata=valores_formatados_piores,
    hovertemplate="Loja: %{x}<br>Faturamento: R$ %{customdata}<extra></extra>"
)

fig_piores.update_yaxes(
    range=[0, faturamento_loja.tail(5).max() * 1.25],  
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    showgrid=True,
    gridcolor='gray',
    griddash='dot',
    ticklabelposition="outside",
    ticklabelstandoff=12,
    ticks="outside",
    title_standoff=25,
    title_font=dict(size=14, color='white')  
)

fig_piores.update_xaxes(
    color='white',
    showline=True,
    linecolor='white',
    linewidth=2,
    tickangle=0,  
    tickfont=dict(size=12, color='white'),  
    title_font=dict(size=14, color='white'),
    title_standoff=35,  
    automargin=True      
)

fig_piores.update_layout(
    margin=dict(t=120, l=120, r=50, b=100), 
    height=550,
    bargap=0.4,  
    plot_bgcolor='#0d1117',
    paper_bgcolor='#0d1117',
    font=dict(color='white', size=12),  
    title_font=dict(size=22, color='white'),  
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="black",
        font_size=12,
        font_color="white"
    ),
    coloraxis_showscale=False
)

fig_piores.show()
